<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 7 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">乱序裁决、历史保留和可恢复重放</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">使用统一订单样本，观察 SQL、结果与验收证据。请按顺序运行单元。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">目标 Doris 4.1.3 · 订单数据 · 独立实验库</span>
</div>

先完成 D06。完成后，你将验证乱序与重复投递后的十一笔当前订单、十八条逻辑历史，并在独立副本上测试部分列更新和删除。请按顺序运行；这不是 Binlog CDC 实验。

[讲义](course7_updates_deletes_and_replay.md) · [课程入口](../README.md)


## 前置条件与业务来源

先完成 D06，本 Lab 从 orders_clean 读取合格新订单，不能绕过准入。
订单号 900001–900011，来源 COURSE_SIMULATION；引用 WWI 客户和商品，但没有改写 WWI 历史。
只重建 orders_current、order_events、event_deliveries、orders_partial_update、orders_delete_demo、order_items、products、payments、refunds、shipment_events。

正常流程：创建 → 支付 → 发货 → 签收；退款流程：创建 → 支付 → 取消 → 退款。
投递次序故意打乱，先收到签收再收到支付。各版本携带完整 after-image。
这是明确的业务事件重放，不是已经接通 Kafka 或 Binlog CDC。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()




## 1. 分开建当前表与历史表

当前表 UNIQUE KEY(order_id) 使用 event_version 裁决；历史表 UNIQUE KEY(event_id) 保留不同业务事件。两者由实验代码显式维护，不假设跨表原子提交。


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_current")
ddl = order_ddl("orders_current", current=True)
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS order_events")
ddl = order_ddl("order_events", history=True)
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.execute(f"INSERT INTO orders_current ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_clean")
lab.execute(f"INSERT INTO order_events ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_clean")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_current"), [(10,"1400.00")])
lab.execute("DROP TABLE IF EXISTS event_deliveries")
lab.execute('CREATE TABLE event_deliveries (attempt_id BIGINT, delivery_id BIGINT, event_id VARCHAR(32), payload STRING) DUPLICATE KEY(attempt_id, delivery_id) DISTRIBUTED BY HASH(attempt_id) BUCKETS 1 PROPERTIES("replication_num"="1")')


## 2. 模拟中断，再恢复

先记录第一条投递并写入历史，然后模拟在当前表写入之前中断。再次投递整批事件，保留每次投递，按稳定事件 ID 去重历史。该实验只中断课程步骤，不终止数据库进程。


In [ ]:
import json
deliveries = fixture("deliveries.json")
first = deliveries[0]
lab.insert("event_deliveries", ["attempt_id","delivery_id","event_id","payload"],
           [(0, first["delivery_id"], first["event_id"], json.dumps(first))])
lab.insert("order_events", ORDER_COLUMNS, order_rows([first]))
expect(lab.query("SELECT status FROM orders_current WHERE order_id=900001"), [("CREATED",)])

def replay(attempt):
    for record in deliveries:
        lab.insert("event_deliveries", ["attempt_id","delivery_id","event_id","payload"],
                   [(attempt, record["delivery_id"], record["event_id"], json.dumps(record))])
        lab.insert("order_events", ORDER_COLUMNS, order_rows([record]))
        lab.insert("orders_current", ORDER_COLUMNS, order_rows([record]))

replay(1)


## 3. 逐行对账，再完整重放

900001 保持版本 4 的 DELIVERED，900003 保持版本 4 的 REFUNDED 且累计支付仍为 150.00。
10 条初始快照加 8 个不同事件，得到 18 条逻辑历史和 11 笔当前订单。
每批 9 次投递含一次重复；中断前 1 次，加两次整批重放，原始投递为 19 次。


In [ ]:
projection = ",".join("DATE_FORMAT(event_time, '%Y-%m-%d %H:%i:%s')" if col == "event_time" else col for col in ORDER_COLUMNS)
def verify_state():
    expect(lab.query(f"SELECT {projection} FROM orders_current ORDER BY order_id"), order_rows(fixture("expected_current.json")))
    expect(lab.query("SELECT COUNT(*), SUM(order_amount), SUM(paid_amount), SUM(refund_amount) FROM orders_current"),
           [(11,"1510.00","250.00","150.00")])
    expect(lab.query("SELECT COUNT(*) FROM order_events"), [(18,)])
    expected_history = {r["event_id"]:r for r in fixture("orders.json") + deliveries}
    expect(lab.query(f"SELECT {projection} FROM order_events ORDER BY event_id"),
           order_rows([expected_history[key] for key in sorted(expected_history)]))
verify_state()
replay(2)
verify_state()
expect(lab.query("SELECT COUNT(*) FROM event_deliveries"), [(19,)])


### 用业务流水独立核对当前状态

不能只拿当前表与当前表比较。用 business_events.json 中独立列出的商品明细、支付、退款和配送事件核对：
商品明细合计应等于订单金额，支付 250.00、退款 150.00；付款和退款 ID 用于业务去重。
退款必须能关联原支付；配送事件必须关联订单。商品维度读取 D05 已导入的 wwi_products（227 行）。同一套业务流水再写两遍不应变多。

这些流水也是课程模拟，与 WWI 的客户账款分开存放。不要拿历史账户收款冒充这里的逐订单付款。


In [ ]:
business = fixture("business_events.json")
definitions = {
    "order_items": 'order_line_id BIGINT NOT NULL, order_id BIGINT, product_id BIGINT, quantity INT, unit_price DECIMAL(12,2)',
    "products": 'product_id BIGINT NOT NULL, product_name STRING',
    "payments": 'payment_id VARCHAR(32) NOT NULL, order_id BIGINT, event_id VARCHAR(32), amount DECIMAL(12,2), event_time DATETIME',
    "refunds": 'refund_id VARCHAR(32) NOT NULL, payment_id VARCHAR(32), order_id BIGINT, event_id VARCHAR(32), amount DECIMAL(12,2), event_time DATETIME',
    "shipment_events": 'shipment_event_id VARCHAR(32) NOT NULL, shipment_id VARCHAR(32), order_id BIGINT, event_id VARCHAR(32), status VARCHAR(20), event_time DATETIME',
}
for table, fields in definitions.items():
    key = fields.split()[0]
    lab.execute("DROP TABLE IF EXISTS " + table)
    ddl = f'CREATE TABLE {table} ({fields}) UNIQUE KEY({key}) DISTRIBUTED BY HASH({key}) BUCKETS 1 PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")'
    show_sql("业务流水表", ddl)
    lab.execute(ddl)
lab.execute("INSERT INTO products SELECT StockItemID, StockItemName FROM wwi_products")
expect(lab.query("SELECT COUNT(*) FROM products"), [(227,)])
for attempt in range(2):
    for table, records in [("order_items", business["order_lines"]), ("payments", business["payments"]),
                           ("refunds", business["refunds"]), ("shipment_events", business["shipments"])]:
        columns = list(records[0])
        lab.insert(table, columns, [tuple(r[c] for c in columns) for r in records])
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM payments"), [(2, "250.00")])
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM refunds"), [(1, "150.00")])
expect(lab.query("SELECT COUNT(*) FROM shipment_events"), [(2,)])
expect(lab.query("SELECT COUNT(*) FROM order_items i LEFT JOIN products p ON i.product_id=p.product_id WHERE p.product_id IS NULL"), [(0,)])
expect(lab.query("SELECT COUNT(*) FROM orders_current o LEFT JOIN customers c ON o.customer_id=c.customer_id WHERE c.customer_id IS NULL"), [(0,)])
expect(lab.query("""
SELECT COUNT(*) FROM refunds r LEFT JOIN payments p
ON r.payment_id=p.payment_id AND r.order_id=p.order_id
WHERE p.payment_id IS NULL OR r.amount>p.amount
"""), [(0,)])
for table in ("payments", "refunds", "shipment_events"):
    expect(lab.query(f"SELECT COUNT(*) FROM {table} b LEFT JOIN order_events h ON b.event_id=h.event_id AND b.order_id=h.order_id WHERE h.event_id IS NULL"), [(0,)])
expect(lab.query("""
SELECT COUNT(*) FROM orders_current o
LEFT JOIN (SELECT order_id, SUM(quantity*unit_price) amount FROM order_items GROUP BY order_id) i ON o.order_id=i.order_id
LEFT JOIN (SELECT order_id, SUM(amount) paid FROM payments GROUP BY order_id) p ON o.order_id=p.order_id
LEFT JOIN (SELECT order_id, SUM(amount) refunded FROM refunds GROUP BY order_id) r ON o.order_id=r.order_id
WHERE i.order_id IS NULL OR o.order_amount<>i.amount
   OR o.paid_amount<>COALESCE(p.paid,0) OR o.refund_amount<>COALESCE(r.refunded,0)
"""), [(0,)])
lab.sql("SELECT order_id, status, event_version, paid_amount, refund_amount, data_source FROM orders_current ORDER BY order_id", title="订单当前状态与来源")


## 4. 部分列更新

仅在独立表上测试；更新版本也要递增。实验记录当前会话的部分更新开关，并在 finally 中恢复。


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_partial_update")
ddl = order_ddl("orders_partial_update", current=True)
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.insert("orders_partial_update", ORDER_COLUMNS, order_rows(fixture("orders.json")[:1]))
previous_partial = lab.query("SELECT @@enable_unique_key_partial_update")[0][0]
try:
    lab.execute("SET enable_unique_key_partial_update = true")
    lab.execute("INSERT INTO orders_partial_update (order_id, status, event_version) VALUES (900001, 'CANCELLED', 2)")
finally:
    lab.execute("SET enable_unique_key_partial_update = %s", (previous_partial,))
expect(lab.query("SELECT status, event_version, order_amount, region FROM orders_partial_update"),
       [("CANCELLED",2,"100.00","EAST")])


## 5. 独立删除实验

主线退款订单不删除。副本中用 is_deleted 表达业务软删除，再用 SQL DELETE 删除另一行；查询不可见不等于磁盘立即回收。


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_delete_demo")
lab.execute('CREATE TABLE orders_delete_demo (order_id BIGINT, is_deleted BOOLEAN, status VARCHAR(20)) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")')
lab.insert("orders_delete_demo", ["order_id","is_deleted","status"], [(900001,False,"CREATED"),(900002,False,"CREATED")])
lab.execute("UPDATE orders_delete_demo SET is_deleted=true WHERE order_id=900001")
expect(lab.query("SELECT COUNT(*) FROM orders_delete_demo"), [(2,)])
expect(lab.query("SELECT order_id FROM orders_delete_demo WHERE is_deleted=false"), [(900002,)])
lab.execute("DELETE FROM orders_delete_demo WHERE order_id=900002")
expect(lab.query("SELECT order_id FROM orders_delete_demo"), [(900001,)])
verify_state()
lab.close()


## 完成与自己动手

验收：11 笔当前订单、18 条逻辑历史、19 次原始投递；逐字段匹配，商品明细和支付退款对账通过。
尝试按 event_time 展示订单 900003 的历史，解释为何退款后 paid_amount 仍为 150.00，而净收款为零。
历史 WWI 表不参与本实验更新。本实验不验证真实 CDC 快照切换、位点恢复或同事件 ID 不同内容的冲突处理。
